### Installtion

In [1]:
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"
!pip install antropy

  Cloning https://github.com/janithcyapa/OpenDatasetStore.git to /tmp/pip-req-build-6wbkz6m_
  Running command git clone --filter=blob:none --quiet https://github.com/janithcyapa/OpenDatasetStore.git /tmp/pip-req-build-6wbkz6m_
  Resolved https://github.com/janithcyapa/OpenDatasetStore.git to commit 8b7a8dbbc28faa53bc5c2de67e562d97d4a89b6f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for open_dataset_store: filename=open_dataset_store-2.0.1-py3-none-any.whl size=24538 sha256=bb6116ef36ba459665fb13465885a5a3f5798717a566469f0fb13aff15773a80
  Stored in directory: /tmp/pip-ephem-wheel-cache-n6ndnxnq/wheels/63/22/ef/d2f6ce9e67edffbf2ce93b9864b71460150ac2e5309ee86676
Successfully built open_dataset_store
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### import

In [4]:
from google.colab import drive
import open_dataset_store as ds
import numpy as np
import pandas as pd
import plotly.express as px
import sims as kf_module
from scipy.spatial.transform import Rotation as R

In [5]:
print('Open Dataset Store Version :',ds.__version__)
drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = ds.quick_start(dataset_path, backend='local')
store.summary('experiments')

Mounted at /content/drive
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics (Backend: local)
📊 Dataset Store Summary
Base Directory : /content/drive/MyDrive/Datasets/Wearable-Biomechanics
Backend        : local
------------------------------
Entities (Total: 3)
  - subjects: 3
------------------------------
Entries (Total: 7)
  - experiments: 6
  - device: 1


{'store_base_dir': '/content/drive/MyDrive/Datasets/Wearable-Biomechanics',
 'backend': 'local',
 'entities': {'subjects': 3},
 'entries': {'experiments': 6, 'device': 1},
 'total_entities': 3,
 'total_entries': 7}

### Get Data

In [9]:
store.list_entities('subjects')

,entity_id,name,age,weight,height,experience,achievement,best_weight,notes
0,sub_0001,Hivindu Kirulapana,22,72,1.74,1,-,-,Participation in Other Sports - Rugby
1,sub_0002,Yasith Dissanayake,22,69,178.00,4,-,55,Participation in Other Sports - Rugby
2,sub_0003,Yewin Dissanayake,22,65,170.00,2,Intermediate bronze medal 2025,95,Participation in Other sports: none


In [ ]:
store.list_entries('experiments')
# store.get_entry("experiments", "experiments_0001")

In [ ]:
df_1 = store.get_entry_raw_data("experiments", "experiments_0001")
df_2 = store.get_entry_raw_data("experiments", "experiments_0003")

In [ ]:
def rotmat_to_quat(R_mat):
    """Convert 3x3 rotation matrix to quaternion [w, x, y, z]"""
    rot = R.from_matrix(R_mat)
    q = rot.as_quat()  # scipy returns [x, y, z, w]
    return np.array([q[3], q[0], q[1], q[2]])  # [w, x, y, z]
    
def DEKF(df,device_prefix):
    accel_cols = [f'{device_prefix}_AX', f'{device_prefix}_AY', f'{device_prefix}_AZ']
    gyro_cols = [f'{device_prefix}_GX', f'{device_prefix}_GY', f'{device_prefix}_GZ']

    accel = df[accel_cols].to_numpy(dtype=float)
    gyro  = df[gyro_cols].to_numpy(dtype=float)
    
    # Normalize accelerometer → unit vector (gravity direction)
    accel_norm = np.linalg.norm(accel, axis=1, keepdims=True)
    accel_norm = np.where(accel_norm < 1e-8, 1.0, accel_norm)
    y_data = accel / accel_norm

    # Filter Configuration
    E = np.array([[0.0, 0.0, -1.0]])
    dt = 0.01
    # Tuning parameters
    Sigma_gyro = (0.02**2) * np.eye(3)   # Gyro noise
    Sigma_bias = (1e-5**2) * np.eye(3)   # Gyro bias random walk
    Sigma_q = np.block([
        [Sigma_gyro, np.zeros((3, 3))],
        [np.zeros((3, 3)), Sigma_bias],
    ])
    Sigma_m = (0.15**2) * np.eye(3)

    kf = kf_module.LinearKF(use_joseph=True, symmetrize=True)
    iekf = kf_module.SO3IMUSensorFusionBiasEKF(kf=kf, inertial_directions=E, dt=dt)
    res = iekf.run_filter(
        R0_plus=np.eye(3),
        b0_plus=np.zeros(3),
        P0=np.diag([1e-2, 1e-2, 1e-2, 1e-4, 1e-4, 1e-4]),
        Omegas=gyro,
        Ys=y_data,
        Sigma_q=Sigma_q,
        Sigma_m=Sigma_m
    )
    
    R_est = res["R_plus"]  # (T, 3, 3)
    # Convert to quaternions
    quats = np.array([rotmat_to_quat(R_mat) for R_mat in R_est])
    
    # Output DataFrame
    out = pd.DataFrame()
    if 'Timestamp' in df.columns:
        out['Timestamp'] = df['Timestamp'].values
    else:
        out['Timestamp'] = np.arange(len(df))
    
    out[f'{device_prefix}_qw'] = quats[:, 0]
    out[f'{device_prefix}_qx'] = quats[:, 1]
    out[f'{device_prefix}_qy'] = quats[:, 2]
    out[f'{device_prefix}_qz'] = quats[:, 3]
    
    return out

def Orientation_Estimation(df, device_list=None):
    """
    Run DEKF on multiple devices and combine results.
    """
    if device_list is None:
        # Auto-detect devices from _AX columns
        accel_cols = [col for col in df.columns if col.endswith('_AX')]
        device_list = sorted(set(col.replace('_AX', '') for col in accel_cols))
    
    results = []
    for dev in device_list:
        print(f"Processing {dev} ...")
        try:
            res = DEKF(df, dev)
            results.append(res)
        except Exception as e:
            print(f"    Error on {dev}: {e}")
    
    if not results:
        raise ValueError("No devices processed successfully.")
    
    # Merge all
    final = results[0]
    for r in results[1:]:
        if 'Timestamp' in r.columns:
            final = pd.merge(final, r, on='Timestamp', how='outer')
        else:
            final = pd.concat([final, r], axis=1)
    
    if 'Timestamp' in final.columns:
        final = final.sort_values('Timestamp').reset_index(drop=True)
    
    return final

In [ ]:
# full_df = Orientation_Estimation(df_1)

In [ ]:
df_1.head()

In [ ]:
fig = px.line(
    df_1,
    x='Timestamp',
    width=1500,
    y=['device1_IMU1_GX','device1_IMU1_GY','device1_IMU1_GZ'],
    title='Device 1 IMU Accelerometer Data Over Time'
)

fig.show(renderer="colab+vscode")

In [ ]:
exp_ids = [f"experiments_{i:04d}" for i in range(1, 7)]
all_data = {exp_id: store.get_entry_raw_data("experiments", exp_id) for exp_id in exp_ids}

### Data Analysis

In [11]:
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

def visualize_selected_columns(df):
    gyro_cols = [col for col in df.columns if col.startswith(('device4')) and col.endswith(('GX','GY', 'GZ'))]
    df['total_gyro'] = df[gyro_cols].abs().sum(axis=1)

    fig = px.line(df, x='Timestamp', y=gyro_cols, width=1500, title='Threshold-Based Segmentation')
    fig.update_traces(visible='legendonly')
    fig.add_scatter(x=df['Timestamp'], y=df['total_gyro'], mode='lines', name='Total', line=dict(color='white', width=1))
    fig.show(renderer="colab+vscode")

for exp_id in all_data.keys():
    print(f"Visualizing {exp_id}...")
    visualize_selected_columns(all_data[exp_id])

Visualizing experiments_0001...


Visualizing experiments_0002...


Visualizing experiments_0003...


Visualizing experiments_0004...


Visualizing experiments_0005...


Visualizing experiments_0006...


### Segmentation Dev

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

def segmentation(df, window=120,threshold=150, config=('device2_IMU1', 'device4_IMU1','device7_IMU1', 'device6_IMU1'), congif_1=('GX','GY', 'GZ')):
    gyro_cols = [col for col in df.columns if col.startswith(config) and col.endswith(congif_1)]
    df['total_gyro'] = df[gyro_cols].abs().sum(axis=1)
    df['moving_avg'] = df['total_gyro'].rolling(window=window, min_periods=1).mean()

# 2. Threshold Triggering
    # Find points where signal crosses above the threshold
    is_active = df['moving_avg'] > threshold
    
    # Identify where the signal switches from False to True (Start of a Rep)
    # and True to False (End of a Rep)
    change = is_active.astype(int).diff()
    starts = df.index[change == 1].tolist()
    ends = df.index[change == -1].tolist()

    # 3. Plotting
    fig = px.line(df, x='Timestamp', y=gyro_cols, width=1500, title='Threshold-Based Segmentation')
    fig.update_traces(visible='legendonly')
    fig.add_scatter(x=df['Timestamp'], y=df['moving_avg'], mode='lines', name='Moving Avg', line=dict(color='red', width=3))
    fig.add_hline(y=threshold, line_dash="dash", line_color="gray", annotation_text="Threshold")


    fig.show(renderer="colab+vscode")

In [ ]:
for exp_id in ["experiments_0004","experiments_0006"]:
    print(f"Processing {exp_id} from cache...")
    segmentation(all_data[exp_id], threshold=100,window=125, config=('device2_IMU1', 'device4_IMU1'),)

In [ ]:
for exp_id in ["experiments_0001", "experiments_0002"]:
    print(f"Processing {exp_id} from cache...")
    segmentation(all_data[exp_id], config=('device10_IMU1',), window=60, threshold=60)

### Segmentation FInal

In [6]:
import plotly.express as px
import pandas as pd
import numpy as np

def extract_segments(df, exp_id, window=120, threshold=150, config=('device2_IMU1', 'device4_IMU1','device7_IMU1', 'device6_IMU1'), config_1=('GX','GY', 'GZ'), plot=True):
    """
    Calculates moving average, finds below-threshold intervals, extracts midpoints, 
    and labels them as repX_segY (up to 5 reps).
    """
    # 1. Moving Average Calculation
    gyro_cols = [col for col in df.columns if col.startswith(config) and col.endswith(config_1)]
    df['total_gyro'] = df[gyro_cols].abs().sum(axis=1)
    df['moving_avg'] = df['total_gyro'].rolling(window=window, min_periods=1).mean()

    # 2. Threshold Triggering
    is_active = df['moving_avg'] > threshold
    change = is_active.astype(int).diff()
    
    # Use integer locations (iloc) for safe midpoint calculation
    starts_iloc = np.where(change == 1)[0]
    ends_iloc = np.where(change == -1)[0]

    # Handle case where data starts already above threshold
    if is_active.iloc[0]:
        starts_iloc = np.insert(starts_iloc, 0, 0)

    # 3. Define Intervals (The "Inactive" gaps)
    intervals = []
    
    # First interval: from index 0 to the first time it crosses ABOVE threshold
    if len(starts_iloc) > 0:
        intervals.append((0, starts_iloc[0]))
        
    # Subsequent intervals: from crossing BELOW to the next crossing ABOVE
    for i in range(min(len(ends_iloc), len(starts_iloc) - 1)):
        intervals.append((ends_iloc[i], starts_iloc[i+1]))

    # 4. Calculate Midpoints and Apply Naming Convention
    segment_markers = {}
    
    for i, (start_idx, end_idx) in enumerate(intervals):
        rep_num = (i // 3) + 1  # Increments every 3 segments
        if rep_num > 5:         # Stop after 5 reps
            break
            
        seg_num = (i % 3) + 1   # Cycles 1, 2, 3
        label = f"rep{rep_num}_seg{seg_num}"
        
        mid_idx = (start_idx + end_idx) // 2
        mid_timestamp = df['Timestamp'].iloc[mid_idx]
        
        segment_markers[label] = {
            'timestamp': mid_timestamp,
            'index': mid_idx,
            'value': df['moving_avg'].iloc[mid_idx]
        }

    # 5. Plotting
    if plot:
        fig = px.line(df, x='Timestamp', y=gyro_cols, width=1500, title=f'Segmentation: {exp_id}')
        fig.update_traces(visible='legendonly')
        fig.add_scatter(x=df['Timestamp'], y=df['moving_avg'], mode='lines', name='Moving Avg', line=dict(color='red', width=3))
        fig.add_hline(y=threshold, line_dash="dash", line_color="gray", annotation_text="Threshold")
        
        # Plot the calculated midpoints to visually verify logic
        for label, data in segment_markers.items():
            fig.add_vline(x=data['timestamp'], line_dash="dot", line_color="green", annotation_text=label, annotation_position="top right")

        fig.show(renderer="colab+vscode")

    return segment_markers

In [7]:
# Dictionary to hold all segment results
experiment_segments = {}

# Process Group 1
for exp_id in ["experiments_0004", "experiments_0006"]:
    print(f"Processing {exp_id} from cache...")
    markers = extract_segments(
        df=all_data[exp_id], 
        exp_id=exp_id,
        threshold=100, 
        window=125, 
        config=('device2_IMU1', 'device4_IMU1')
    )
    experiment_segments[exp_id] = markers

# Process Group 2
for exp_id in ["experiments_0001", "experiments_0002"]:
    print(f"Processing {exp_id} from cache...")
    markers = extract_segments(
        df=all_data[exp_id], 
        exp_id=exp_id,
        config=('device10_IMU1',), 
        window=60, 
        threshold=75
    )
    experiment_segments[exp_id] = markers

# Example of how to access the extracted data later:


Processing experiments_0004 from cache...


Processing experiments_0006 from cache...


Processing experiments_0001 from cache...


Processing experiments_0002 from cache...


### Entropy

In [4]:
import pandas as pd
import numpy as np
import antropy as ant
import plotly.express as px
import itertools

exp_ids = [f"experiments_{i:04d}" for i in range(1, 7)]
all_data = {exp_id: store.get_entry_raw_data("experiments", exp_id) for exp_id in exp_ids}

def calculate_sampen(time_series):
    """
    Calculates Sample Entropy for IMU time-series data.
    """
    clean_series = time_series.dropna().values
    
    # Safety check: SampEn needs a minimum data length to be valid.
    # At 100Hz, anything under 20 frames (0.2 seconds) is likely a glitch/noise.
    if len(clean_series) < 20: 
        return np.nan
        
    # Calculate and return Sample Entropy
    # Defaults: m=2, r=0.2 * std(data)
    return ant.sample_entropy(clean_series)

target_exps = ['experiments_0001', 'experiments_0002', 'experiments_0004', 'experiments_0006']
ignore_cols = ['Timestamp', 'total_gyro', 'moving_avg']

### Segment Wise

In [12]:
# Set up the Storage List
entropy_results = []

# Loop through all processed experiments in your dictionary
for exp_id, markers in experiment_segments.items():
    print(f"Calculating Entropy for {exp_id}...")
    
    df = all_data[exp_id]
    
    # Identify the columns we actually want to calculate entropy for
    target_columns = [col for col in df.columns if col not in ignore_cols]
    
    # Sort markers chronologically by their index to ensure correct pairing
    sorted_markers = sorted(markers.items(), key=lambda x: x[1]['index'])
    
    # Iterate through markers pairwise to create the active segments
    for i in range(len(sorted_markers) - 1):
        start_label, start_data = sorted_markers[i]
        
        # We only need the end data for the slicing index now, not the label
        _, end_data = sorted_markers[i+1] 
        
        # Slice the dataframe for this specific segment
        start_idx = start_data['index']
        end_idx = end_data['index']
        segment_df = df.iloc[start_idx:end_idx]
        
        # Create a dictionary for this row's metadata using ONLY the start label
        row_data = {
            'Experiment': exp_id,
            'Segment_Name': start_label,
            'Duration_Frames': end_idx - start_idx
        }
        
        # 4. Calculate SampEn for every target column in this segment
        for col in target_columns:
            # Append a suffix so you know these are SampEn values
            col_name = f"{col}_SampEn" 
            row_data[col_name] = calculate_sampen(segment_df[col])
            
        # Append the completed row to our results list
        entropy_results.append(row_data)

# Convert to a final DataFrame
df_entropy = pd.DataFrame(entropy_results)
df_entropy


Calculating Entropy for experiments_0004...
Calculating Entropy for experiments_0006...
Calculating Entropy for experiments_0001...
Calculating Entropy for experiments_0002...


,Experiment,Segment_Name,Duration_Frames,device1_IMU1_AX_SampEn,device1_IMU1_AY_SampEn,device1_IMU1_AZ_SampEn,device1_IMU1_GX_SampEn,device1_IMU1_GY_SampEn,device1_IMU1_GZ_SampEn,device1_IMU2_AX_SampEn,...,device10_IMU2_AZ_SampEn,device10_IMU2_GX_SampEn,device10_IMU2_GY_SampEn,device10_IMU2_GZ_SampEn,device10_IMU3_AX_SampEn,device10_IMU3_AY_SampEn,device10_IMU3_AZ_SampEn,device10_IMU3_GX_SampEn,device10_IMU3_GY_SampEn,device10_IMU3_GZ_SampEn
0,experiments_0004,rep1_seg1,1102,0.081448,0.063606,0.099845,0.105015,0.100936,0.070100,0.046572,...,0.047924,0.088902,0.057841,0.040598,0.066057,0.050178,0.096744,0.103569,0.077649,0.050642
1,experiments_0004,rep1_seg2,448,0.285383,0.415191,0.489648,0.356692,0.302339,0.276516,0.243463,...,0.249021,0.241035,0.142162,0.136839,0.142076,0.422396,0.359811,0.301347,0.212178,0.105310
2,experiments_0004,rep1_seg3,764,0.722354,0.440028,0.389940,0.369417,0.674267,0.444598,0.627237,...,0.484386,0.187275,0.187713,0.202864,0.450022,0.426857,0.551599,0.302703,0.247905,0.141500
3,experiments_0004,rep2_seg1,703,0.188241,0.213828,0.324994,0.273249,0.192540,0.121449,0.135656,...,0.250007,0.213616,0.200684,0.096715,0.182444,0.209055,0.232543,0.177949,0.211123,0.104245
4,experiments_0004,rep2_seg2,454,0.148453,0.296471,0.341042,0.286664,0.269060,0.206609,0.111712,...,0.124326,0.121076,0.127709,0.127171,0.126877,0.252276,0.253079,0.200565,0.112459,0.115365
5,experiments_0004,rep2_seg3,769,0.762020,0.352524,0.401175,0.409390,0.466175,0.398484,0.653822,...,0.404758,0.263276,0.242717,0.205450,0.519000,0.446486,0.501671,0.323373,0.273530,0.195494
6,experiments_0004,rep3_seg1,672,0.207549,0.293632,0.350365,0.316836,0.245153,0.158285,0.130791,...,0.269082,0.238294,0.218849,0.127486,0.211132,0.217435,0.304175,0.228976,0.188414,0.139518
7,experiments_0004,rep3_seg2,463,0.284326,0.477809,0.588593,0.452013,0.350992,0.223792,0.257573,...,0.164264,0.192845,0.210450,0.127378,0.181685,0.400887,0.364989,0.344046,0.202784,0.138064
8,experiments_0004,rep3_seg3,754,0.625341,0.501601,0.495838,0.401929,0.485346,0.336280,0.703163,...,0.289117,0.271641,0.181181,0.172953,0.313162,0.396030,0.310947,0.256714,0.189250,0.160415
9,experiments_0004,rep4_seg1,620,0.266972,0.245868,0.257344,0.212352,0.195891,0.138071,0.180127,...,0.221063,0.161940,0.158453,0.084566,0.161877,0.192802,0.228268,0.161844,0.153095,0.117567


In [13]:
# Generate ALL expected segments (rep1_seg1 through rep5_seg3)
all_segments = [f"rep{r}_seg{s}" for r in range(1, 6) for s in range(1, 4)]

# Create a Master Grid of every Experiment + Segment combination
master_grid = pd.DataFrame(
    list(itertools.product(target_exps, all_segments)), 
    columns=['Experiment', 'Segment_Name']
)

# Filter original data and merge it into the Master Grid
df_actual = df_entropy[df_entropy['Experiment'].isin(target_exps)].copy()

# A left merge keeps all 60 rows from the master grid, inserting NaN where data is missing
df_plot = pd.merge(master_grid, df_actual, on=['Experiment', 'Segment_Name'], how='left')

# Identify SampEn columns and fill missing (NaN) values with 0
sampen_cols = [col for col in df_plot.columns if str(col).endswith('_SampEn')]
df_plot[sampen_cols] = df_plot[sampen_cols].fillna(0)

# Apply Categorical sorting so Plotly respects your exact order
df_plot['Experiment'] = pd.Categorical(df_plot['Experiment'], categories=target_exps, ordered=True)
df_plot['Segment_Name'] = pd.Categorical(df_plot['Segment_Name'], categories=all_segments, ordered=True)
df_plot.sort_values(by=['Experiment', 'Segment_Name'], inplace=True)

# Create unique Y-axis labels 
df_plot['Y_Label'] = df_plot['Experiment'].astype(str).str.replace('experiments_', 'Exp ') + " - " + df_plot['Segment_Name'].astype(str)
df_plot.set_index('Y_Label', inplace=True)

# Extract just the heatmap data
df_heatmap = df_plot[sampen_cols]

# Find row indices for separator lines
exp_changes = []
current_exp = df_plot['Experiment'].iloc[0]

for i, exp in enumerate(df_plot['Experiment']):
    if exp != current_exp:
        exp_changes.append(i - 0.5) 
        current_exp = exp

# Build the Plot
fig = px.imshow(
    df_heatmap,
    labels=dict(x="IMU Features", y="Experiment & Segment", color="Sample Entropy"),
    title="Sample Entropy Signatures Across Experiments (Missing Filled w/ 0)",
    aspect="auto",
    color_continuous_scale="Viridis",
    height=1200, # Slightly taller to comfortably fit all 60 guaranteed rows
    text_auto=".2f"
)

# Move X-axis labels to the top for readability
fig.update_xaxes(side="top")

# Add the horizontal separator lines
for y_pos in exp_changes:
    fig.add_hline(
        y=y_pos, 
        line_width=2, 
        line_dash="dash", 
        line_color="white"
    )

fig.show(renderer="colab+vscode")

/tmp/ipykernel_1127/4048548571.py:26: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



# Rep Wise

In [14]:
import pandas as pd
import numpy as np
import plotly.express as px
import antropy as ant
import itertools

# 1. Setup Storage
rep_entropy_results = []
target_exps = ['experiments_0001', 'experiments_0002', 'experiments_0004', 'experiments_0006']
ignore_cols = ['Timestamp', 'total_gyro', 'moving_avg']

# 2. Recalculate for Full Repetitions (Only for actual existing data)
for exp_id in target_exps:
    df = all_data[exp_id]
    markers = experiment_segments[exp_id]
    target_columns = [col for col in df.columns if col not in ignore_cols]
    
    # Sort markers chronologically
    sorted_markers = sorted(markers.items(), key=lambda x: x[1]['index'])
    
    # Extract unique rep numbers (e.g., 'rep1', 'rep2', etc.)
    reps = sorted(list(set([label.split('_')[0] for label, data in sorted_markers])))
    
    for i, rep in enumerate(reps):
        # Find the first segment of this rep
        start_marker = next(data for label, data in sorted_markers if label.startswith(f"{rep}_seg1"))
        start_idx = start_marker['index']
        
        # End index logic (Start of next rep, or end of dataset)
        if i + 1 < len(reps):
            next_rep = reps[i+1]
            end_marker = next((data for label, data in sorted_markers if label.startswith(f"{next_rep}_seg1")), None)
            if end_marker:
                end_idx = end_marker['index']
            else:
                end_idx = sorted_markers[-1][1]['index']
        else:
            end_idx = sorted_markers[-1][1]['index'] 
            
        # Slice raw data
        rep_df = df.iloc[start_idx:end_idx]
        
        row_data = {'Experiment': exp_id, 'Repetition': rep}
        
        for col in target_columns:
            clean_series = rep_df[col].dropna().values
            if len(clean_series) >= 20:
                row_data[f"{col}_SampEn"] = ant.sample_entropy(clean_series)
            else:
                row_data[f"{col}_SampEn"] = np.nan
                
        rep_entropy_results.append(row_data)

# Actual calculated data
df_actual_reps = pd.DataFrame(rep_entropy_results)

# 3. Create Master Grid to enforce exactly 5 reps per experiment (rep1 to rep5)
all_reps = [f"rep{i}" for i in range(1, 6)]
master_grid = pd.DataFrame(
    list(itertools.product(target_exps, all_reps)), 
    columns=['Experiment', 'Repetition']
)

# 4. Merge and Fill Missing with 0
df_plot_reps = pd.merge(master_grid, df_actual_reps, on=['Experiment', 'Repetition'], how='left')

# Identify SampEn columns and fill missing (NaN) values with 0
sampen_cols = [col for col in df_plot_reps.columns if str(col).endswith('_SampEn')]
df_plot_reps[sampen_cols] = df_plot_reps[sampen_cols].fillna(0)

# 5. Apply Categorical Sorting to maintain exact order
df_plot_reps['Experiment'] = pd.Categorical(df_plot_reps['Experiment'], categories=target_exps, ordered=True)
df_plot_reps['Repetition'] = pd.Categorical(df_plot_reps['Repetition'], categories=all_reps, ordered=True)
df_plot_reps.sort_values(by=['Experiment', 'Repetition'], inplace=True)

# 6. Format labels for Y-Axis (e.g., "Exp 0001 - rep1")
df_plot_reps['Y_Label'] = df_plot_reps['Experiment'].astype(str).str.replace('experiments_', 'Exp ') + " - " + df_plot_reps['Repetition'].astype(str)
df_plot_reps.set_index('Y_Label', inplace=True)

# 7. Find Row Indices for Separator Lines
exp_changes = []
current_exp = df_plot_reps['Experiment'].iloc[0]

for i, exp in enumerate(df_plot_reps['Experiment']):
    if exp != current_exp:
        exp_changes.append(i - 0.5) 
        current_exp = exp

# 8. Plot Repetition Heatmap
fig_reps = px.imshow(
    df_plot_reps[sampen_cols],
    labels=dict(x="IMU Features", y="Experiment & Repetition", color="Sample Entropy"),
    title="Sample Entropy Per Full Repetition (Missing Filled w/ 0)",
    aspect="auto",
    color_continuous_scale="Viridis", # Unified theme
    height=800,
    text_auto=".2f"
)

# Move X-axis labels to the top
fig_reps.update_xaxes(side="top")

# Add the horizontal separator lines
for y_pos in exp_changes:
    fig_reps.add_hline(
        y=y_pos, 
        line_width=2, 
        line_dash="dash", 
        line_color="white"
    )

fig_reps.show(renderer="colab+vscode")

/tmp/ipykernel_1127/2130028481.py:77: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



#### Whole Dataset

In [ ]:
# Setup Storage
exp_entropy_results = []

# Recalculate for the Entire Experiment
for exp_id in target_exps:
    df = all_data[exp_id]
    target_columns = [col for col in df.columns if col not in ignore_cols]
    
    row_data = {'Experiment': exp_id}
    
    for col in target_columns:
        clean_series = df[col].dropna().values
        if len(clean_series) >= 20:
            row_data[f"{col}_SampEn"] = ant.sample_entropy(clean_series)
        else:
            row_data[f"{col}_SampEn"] = np.nan
            
    exp_entropy_results.append(row_data)

# Create DataFrame
df_exp_entropy = pd.DataFrame(exp_entropy_results)

df_plot_exp = df_exp_entropy[df_exp_entropy['Experiment'].isin(target_exps)].copy()

# 2. Apply Categorical Sorting to maintain exact order
df_plot_exp['Experiment'] = pd.Categorical(df_plot_exp['Experiment'], categories=target_exps, ordered=True)
df_plot_exp.sort_values(by='Experiment', inplace=True)

# 3. Format Y-axis labels cleanly (e.g., "Exp 0001")
df_plot_exp['Y_Label'] = df_plot_exp['Experiment'].astype(str).str.replace('experiments_', 'Exp ')
df_plot_exp.set_index('Y_Label', inplace=True)

# 4. Extract just the SampEn columns
sampen_cols = [col for col in df_plot_exp.columns if str(col).endswith('_SampEn')]
df_heatmap_exp = df_plot_exp[sampen_cols]

# 5. Plot Global Heatmap
fig_global = px.imshow(
    df_heatmap_exp,
    labels=dict(x="IMU Features", y="Experiment", color="Sample Entropy"),
    title="Global Sample Entropy (Entire Experiment Duration)",
    aspect="auto",
    color_continuous_scale="Viridis", # Unified theme
    height=400, # Shorter height because it is only 4 rows of data
    text_auto=".2f"
)

# Move X-axis labels to the top for readability
fig_global.update_xaxes(side="top")

# Optional: Clean up X-axis labels to remove the redundant '_SampEn' text for a cleaner look
clean_x_labels = [col.replace('_SampEn', '') for col in sampen_cols]
fig_global.update_xaxes(ticktext=clean_x_labels, tickvals=sampen_cols)

fig_global.show(renderer="colab+vscode")

### Segments Combined

In [6]:
all_phases = ['seg1', 'seg2', 'seg3']

# 2. Create Master Grid to guarantee all phases exist for all experiments
master_grid = pd.DataFrame(
    list(itertools.product(target_exps, all_phases)), 
    columns=['Experiment', 'Phase']
)

# 3. Calculate actual phase averages
df_plot_phase = df_entropy[df_entropy['Experiment'].isin(target_exps)].copy()
df_plot_phase['Phase'] = df_plot_phase['Segment_Name'].str.split('_').str[1]
df_actual_phase = df_plot_phase.groupby(['Experiment', 'Phase']).mean(numeric_only=True).reset_index()

# 4. Merge into Master Grid and fill missing with 0
df_phase_avg = pd.merge(master_grid, df_actual_phase, on=['Experiment', 'Phase'], how='left')

# Identify SampEn columns based on the original dataframe
sampen_cols = [col for col in df_plot_phase.columns if str(col).endswith('_SampEn')]
df_phase_avg[sampen_cols] = df_phase_avg[sampen_cols].fillna(0)

# 5. Apply Categorical Sorting to ensure chronological phase order
df_phase_avg['Experiment'] = pd.Categorical(df_phase_avg['Experiment'], categories=target_exps, ordered=True)
df_phase_avg['Phase'] = pd.Categorical(df_phase_avg['Phase'], categories=all_phases, ordered=True)
df_phase_avg.sort_values(by=['Experiment', 'Phase'], inplace=True)

# 6. Format Y-axis labels (e.g., "Exp 0001 - seg1")
df_phase_avg['Y_Label'] = df_phase_avg['Experiment'].astype(str).str.replace('experiments_', 'Exp ') + " - " + df_phase_avg['Phase'].astype(str)
df_phase_avg.set_index('Y_Label', inplace=True)

# 7. Extract just the SampEn columns for the heatmap
df_heatmap_phase = df_phase_avg[sampen_cols]

# 8. Find Row Indices for Separator Lines (to divide experiments)
exp_changes = []
current_exp = df_phase_avg['Experiment'].iloc[0]

for i, exp in enumerate(df_phase_avg['Experiment']):
    if exp != current_exp:
        exp_changes.append(i - 0.5) 
        current_exp = exp

# 9. Build the Plot
fig_phase = px.imshow(
    df_heatmap_phase,
    labels=dict(x="IMU Features", y="Experiment & Movement Phase", color="Avg Sample Entropy"),
    title="Average Sample Entropy by Movement Phase (Missing Filled w/ 0)",
    aspect="auto",
    color_continuous_scale="Viridis",
    height=600, # 12 rows fit nicely in 600px
    text_auto=".2f"
)

# Move X-axis labels to the top for readability
fig_phase.update_xaxes(side="top")

# Optional: Clean up X-axis labels to remove the redundant '_SampEn' text
clean_x_labels = [col.replace('_SampEn', '') for col in sampen_cols]
fig_phase.update_xaxes(ticktext=clean_x_labels, tickvals=sampen_cols)

# Add the horizontal separator lines
for y_pos in exp_changes:
    fig_phase.add_hline(
        y=y_pos, 
        line_width=2, 
        line_dash="dash", 
        line_color="white"
    )

fig_phase.show(renderer="colab+vscode")

NameError: name 'itertools' is not defined